# Étape 2 - Feature engineering sur le jeu d'entraînement (offline)

Un modèle ne sait pas lire une date. Il faut transformer `order_date` en **variables**
chiffrées exploitables. C'est le *feature engineering*.

On parle de mode **offline** car on travaille sur l'historique complet : on a toutes les
données passées sous la main pour calculer un décalage temporel (`lag`).

## Les 3 briques (déjà codées)

| Fonction | Rôle |
|---|---|
| `dummy_day` | encode le jour de la semaine en 6 colonnes binaires (`day_1` … `day_6`, le lundi est la référence) |
| `hour_cos_sin` | encode l'heure sur un cercle : `hour_cos_1` et `hour_sin_1` (ainsi 23h est proche de 0h) |
| `lag_offline` | ajoute `lag_1W` = le `cash_in` d'il y a exactement 1 semaine (`shift` de 7 jours) |

Elles sont enchaînées par la fonction maître **`features_offline`**.

## 1. Importer les librairies

In [3]:
import sys
sys.path.append('..')
import yaml
import logging
import logging.config
import numpy as np
import pandas as pd
pd.set_option('display.min_rows', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 500)
pd.set_option('max_colwidth', 400)

from foodcast.domain.transform import etl
from foodcast.domain.feature_engineering import features_offline, features_online
from foodcast.domain.forecast import span_future, cross_validate, plotly_predictions
from foodcast.domain.multi_model import MultiModel
from sklearn.ensemble import RandomForestRegressor
import foodcast.settings as settings
import plotly.graph_objects as go

with open(settings.LOGGING_CONFIGURATION_FILE, 'r') as f:
    logging.config.dictConfig(yaml.safe_load(f.read()))

%load_ext autoreload
%autoreload 2

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/mlflow/pyfunc/utils/data_validation.py:156: FutureWarning: Model's `predict` method contains invalid parameters: {'X'}. Only the following parameter names are allowed: context, model_input, and params. Note that invalid parameters will no longer be permitted in future versions.
  param_names = _check_func_signature(func, "predict")


************************************************************
USING default value : foodcast.settings.dev
************************************************************


## 2. Reprendre le jeu de données de l'étape 1

In [4]:
df = etl(settings.DATA_DIR, 197, 200)
df.head()

2026-09-09 15:05:27 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/infrastructure/extract.py - INFO - extract: shape = (2158, 6)
2026-09-09 15:05:27 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/infrastructure/extract.py - INFO - extract: shape = (3247, 6)
2026-09-09 15:05:27 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - clean: shape = (380, 3)
2026-09-09 15:05:27 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - clean: shape = (565, 3)
2026-09-09 15:05:27 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - merge: shape = (945, 2)
2026-09-09 15:05:27 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - resample: shape = (659, 2)
2026-09-09 15:05:27 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - IN

,order_date,cash_in
0,2018-10-08 09:00:00,89.55
1,2018-10-08 10:00:00,0.00
2,2018-10-08 11:00:00,0.00
3,2018-10-08 12:00:00,0.00
4,2018-10-08 13:00:00,0.00


## 3. Regarder le code de `features_offline`

In [5]:
features_offline??

Signature:
features_offline(
    df: pandas.DataFrame,
    degree: int = 1,
    lag_in_week: int = 1,
) -> pandas.DataFrame
Source:   
@log_return_shape
def features_offline(df: pd.DataFrame, degree: int = 1, lag_in_week: int = 1) -> pd.DataFrame:
    """
    Offline feature engineering with enough history to compute lags.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe to add features on.
    degree : int, optional
        Degree of the sines and cosines computed, by default 1.
    lag_in_week : int, optional
        Number of weeks to lag, by default 1.

    Returns
    -------
    pd.DataFrame
        Input dataframe with additional features.
    """
    df = dummy_day(df)
    df = hour_cos_sin(df, degree=degree)
    df = lag_offline(df, lag_in_week=lag_in_week)
    return df
File:      ~/Documents/ml_data_base/MlOps_1/foodcast/domain/feature_engineering.py
Type:      function

## 4. Appliquer le feature engineering

On remplace `df` par sa version enrichie.

In [ ]:
df = features_offline(df)
df.head(20)

Nouvelles colonnes :

- `day_1` … `day_6` : jour de la semaine (one-hot, lundi = tout à 0)
- `hour_cos_1`, `hour_sin_1` : heure de la journée encodée en continu
- `lag_1W` : chiffre d'affaires de la même heure, 7 jours plus tôt

> Remarque : les 7 premiers jours disparaissent (`dropna`), car ils n'ont pas de passé à J-7.

In [ ]:
df.shape

## 5. Vérifier `lag_1W` à la main

On prend une heure donnée et on vérifie que `lag_1W` vaut bien le `cash_in`
de la même heure 7 jours avant.

In [ ]:
ligne = df[df['order_date'] == '2018-11-02 19:00:00']
ligne[['order_date', 'cash_in', 'lag_1W']]

In [ ]:
semaine_avant = df[df['order_date'] == '2018-10-26 19:00:00']
semaine_avant[['order_date', 'cash_in']]  # son cash_in doit être égal au lag_1W ci-dessus

## 6. Séparer variables explicatives (X) et cible (y)

On garde la date dans l'**index** (utile pour les graphes et la validation temporelle),
et on met de côté la colonne `cash_in` comme cible.

In [ ]:
x_train = df.drop(columns=['cash_in'])
y_train = df[['order_date', 'cash_in']]
x_train = x_train.set_index('order_date')
y_train = y_train.set_index('order_date')['cash_in']

x_train.head()

In [ ]:
y_train.head()

`x_train` = les features (indexées par date), `y_train` = la série du chiffre d'affaires.

➡️ Étape suivante : `03_entrainement_modele.ipynb`